# T0 — Baseline Gap: stock SmolLM2 vs HAGI from-scratch B

**Question:** how far is a *from-scratch* 114M HAGI model (B = recurrence-only,
held-out CE **4.2581** on `shard_00006`) from a small *pretrained* open model
scored on the **exact same** held-out tokens?

If the gap is large (it will be), it confirms the bitter lesson: a pretrained base
starts from a far better place than any from-scratch run free compute can reach.
That makes *adapter vs from-scratch* a foregone win — and reframes the real test as
**Clifford-adapter vs plain-LoRA at equal budget** (that is T1).

~3–5 min on a free T4. No training. No repo clone needed.


In [ ]:
# Colab usually ships torch. Add transformers + hub if missing.
!pip -q install -U "transformers>=4.44" huggingface_hub numpy
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())


In [ ]:
# --- knobs ---
BASE_MODEL  = "HuggingFaceTB/SmolLM2-360M"        # swap -1.7B for the real pivot base
DATA_REPO   = "NAME0x0/hagi-fineweb-edu-smollm2"  # public HF dataset (no token needed)
VAL_SHARD   = "shard_00006.bin"                   # held-out shard B never trained on
SEQ         = 1024     # match the cloud B eval exactly
BATCH       = 16
N_BATCHES   = 50
SEED        = 1234     # fixes the identical batch set B was scored on
HAGI_B_LOSS = 4.2581   # from-scratch recurrence-only held-out CE (docs/GEO_DIAG.md)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Pull just the held-out shard from the public HF dataset.
from huggingface_hub import hf_hub_download
val_path = hf_hub_download(DATA_REPO, VAL_SHARD, repo_type="dataset")
print("held-out shard:", val_path)


In [ ]:
# Identical batch sampler, inlined from prototype/data/dataset.py:MemmapTokenDataset.
# Replicates the draw order EXACTLY (one shard -> the shard-index draw is still
# consumed to keep the rng in sync), so these are the SAME batches B was scored on.
# uint16 token stream, y = x shifted by one.
import numpy as np
mm = np.memmap(val_path, dtype=np.uint16, mode="r")
N_SHARDS = 1
hi = len(mm) - SEQ - 1
rng = np.random.default_rng(SEED)

def draw_batch():
    xs, ys = [], []
    for _ in range(BATCH):
        _si   = int(rng.integers(N_SHARDS))   # consume shard draw (matches repo order)
        start = int(rng.integers(hi))
        chunk = mm[start:start + SEQ + 1].astype(np.int64)
        xs.append(chunk[:-1]); ys.append(chunk[1:])
    x = torch.from_numpy(np.stack(xs)).to(DEVICE)
    y = torch.from_numpy(np.stack(ys)).to(DEVICE)
    return x, y

batches = [draw_batch() for _ in range(N_BATCHES)]
tok = N_BATCHES * BATCH * SEQ
print(f"eval set: {N_BATCHES} x {BATCH} x {SEQ} = {tok:,} tokens (seed {SEED})")
print(f"shard tokens: {len(mm):,} | max id seen: {int(max(b[0].max() for b in batches))} (vocab 49152)")


In [ ]:
# Load the stock pretrained base and score CE on the identical batches.
import math
import torch.nn.functional as F
from transformers import AutoModelForCausalLM

DT = torch.float16 if DEVICE == "cuda" else torch.float32
try:                      # transformers >=5 renamed `torch_dtype` -> `dtype`
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DT)
except TypeError:         # older transformers (<4.56) still want `torch_dtype`
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DT)
model = model.to(DEVICE).eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"{BASE_MODEL}: {n_params:.1f}M params, dtype {next(model.parameters()).dtype}")

total = 0.0
with torch.no_grad():
    for x, y in batches:
        logits = model(x).logits                                  # [B, T, V]
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(),
                               y.reshape(-1))                      # mean token CE (matches HAGI)
        total += loss.item()
avg = total / len(batches)
ppl = math.exp(avg)

print("
=== T0 RESULT (same tokens, same tokenizer) ===")
print(f"{BASE_MODEL:<32} CE {avg:.4f} | ppl {ppl:8.2f}")
print(f"{'HAGI B (from scratch, 114M)':<32} CE {HAGI_B_LOSS:.4f} | ppl {math.exp(HAGI_B_LOSS):8.2f}")
gap = HAGI_B_LOSS - avg
print(f"
GAP (B - base) = {gap:+.4f} nats   ({'base WINS' if gap > 0 else 'B wins'})")
if gap > 0:
    print(f"perplexity: base is {math.exp(HAGI_B_LOSS)/ppl:.2f}x lower than B")


## Read

- `GAP = B − base`. **Positive = the pretrained base beats our from-scratch B on B's
  own held-out data** — despite B training on exactly this distribution and the base
  never seeing it. That is the trillions-of-tokens head start.
- The bigger the gap, the more decisively *adapter-on-a-base ≫ from-scratch*. That is
  why the real experiment (T1) is **Clifford-adapter vs plain-LoRA at equal budget**,
  not adapter-vs-scratch (which this cell already settles).
- **Apples-to-apples caveat:** both numbers are mean token-level CE on the same
  uint16 SmolLM2-tokenized shard. All SmolLM2 sizes share one tokenizer (49152), so
  the token ids feed the base directly — no re-tokenization.

**Next — T1:** LoRA vs geometric-product adapter on a chosen base + reasoning task,
equal params/budget, held-out gate. Only if the Clifford adapter clears the LoRA bar
do we port the fuller grade/loop variants (T2).
